# Robotics_02

### Виконав: Давидчук Артем, ІО-41

Варіант: 106

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact, IntSlider, FloatSlider

L1 = 0.4
L2 = 0.4
L3 = 0.3
L4 = 0.1

x_5 = 0.8
y_5 = 0.5
alpha_g = -90
theta4_g = -90

def two_link_IK(x, y):
    """
    Calculate inverse kinematics for a two-link planar manipulator.

    Parameters:
    x, y: Target end-effector position
    L1, L2: Lengths of the two links

    Returns:
    (theta1, theta2): Joint angles in radians (two solutions)
    """
    # Calculate distance to target
    d = np.sqrt(x**2 + y**2)
    #print (L1,L2,d,x,y)

    # Check if target is reachable
    if d > L1 + L2 or d < abs(L1 - L2):
        return None, None, None, None  # Target not reachable

    # Calculate theta2 using law of cosines
    cos_theta2 = (x**2 + y**2 - L1**2 - L2**2) / (2 * L1 * L2)
    # Handle numerical errors that might push cos_theta2 beyond [-1, 1]
    cos_theta2 = np.clip(cos_theta2, -1, 1)
    sin_theta2 = np.sqrt(1 - cos_theta2**2)

    # Two solutions for theta2 (elbow up and elbow down)
    theta2_1 = np.arctan2(sin_theta2, cos_theta2)
    theta2_2 = np.arctan2(-sin_theta2, cos_theta2)

    # Calculate theta1 for each solution
    theta1_1 = np.arctan2(y, x) - np.arctan2(L2 * np.sin(theta2_1), L1 + L2 * np.cos(theta2_1))
    theta1_2 = np.arctan2(y, x) - np.arctan2(L2 * np.sin(theta2_2), L1 + L2 * np.cos(theta2_2))

    #return (theta1_1, theta2_1), (theta1_2, theta2_2)
    return theta1_1, theta2_1, theta1_2, theta2_2


def plot_segment_IK4(x5, y5, theta4g, alphag):
    # Перетворюємо кути в радіани
    theta4 = np.radians(theta4g)
    alpha = np.radians(alphag)
    #print (alpha, theta4, np.cos(alpha - theta4))

    # Розраховуємо координати суглобів 3,4 (joints)
    x4 = x5 - L4 * np.cos(alpha)
    y4 = y5 - L4 * np.sin(alpha)
    x3 = x4 - L3 * np.cos(alpha - theta4)
    y3 = y4 - L3 * np.sin(alpha - theta4)

    # IK:
    theta1_1, theta2_1, theta1_2, theta2_2 = two_link_IK(x3, y3)
    # main solution:
    theta1 = theta1_2
    theta2 = theta2_2
    # alt. solution:
    theta1a = theta1_1
    theta2a = theta2_1

    theta3 = alpha - (theta1  + theta2  + theta4)
    theta3a= alpha - (theta1a + theta2a + theta4)

    # Розраховуємо координати суглобів 1,2 (joints)
    x1, y1 = 0, 0
    x2 = x1 + L1 * np.cos(theta1)   # theta1_2 or theta1_1
    y2 = y1 + L1 * np.sin(theta1)   # theta1_2 or theta1_1
    x2a = x1 + L1 * np.cos(theta1a) # theta1_1 or theta1_2
    y2a = y1 + L1 * np.sin(theta1a) # theta1_1 or theta1_2


    theta_1g  = theta1 *180/np.pi
    theta_1ag = theta1a*180/np.pi
    theta_2g  = theta2 *180/np.pi
    theta_2ag = theta2a*180/np.pi
    theta_3g  = theta3 *180/np.pi
    theta_3ag = theta3a*180/np.pi
    theta_4g  = theta4 *180/np.pi

    # Рисуємо манипулятор
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.grid(True)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')

    ax.plot([x1, x2], [y1, y2], 'b-', linewidth=8, label='L1')
    ax.plot([x2, x3], [y2, y3], 'g-', linewidth=5, label='L2')
    ax.plot([x3, x4], [y3, y4], 'r-', linewidth=4, label='L3')
    ax.plot([x4, x5], [y4, y5], 'k-', linewidth=2, label='L4')

    ax.plot([x1, x2a], [y1, y2a], 'b--', linewidth=1, label='L1a') # alt-solition
    ax.plot([x2a, x3], [y2a, y3], 'g--', linewidth=1, label='L2a') #

    ax.plot(x1, y1, 'ro', markersize=9, label='joint 1')
    ax.plot(x2, y2, 'yo', markersize=9, label='joint 2')
    ax.plot(x3, y3, 'ko', markersize=9, label='joint 3')
    ax.plot(x4, y4, 'go', markersize=9, label='joint 4')
    ax.plot(x5, y5, 'rx', markersize=9, label='end effector')


    # Додаємо легенду
    ax.legend(loc='upper right')

    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(0, 1)
    ax.set_title(f'Four-link planar manipulator IK problem:\n Joint1 (θ1={theta_1g:.1f}, θ1a={theta_1ag:.1f}), Joint2 (θ2={theta_2g:.1f}, θ2a={theta_2ag:.1f}), Joint3 (θ3={theta_3g:.1f}, θ3a={theta_3ag:.1f}),  \n Joint4 (θ4={theta_4g:.1f}) end effector (x5={x5:.2f}, y5={y5:.2f}) ')
    plt.show()
    return


#interact(plot_segment, length=IntSlider(min=1, max=10, step=1, value=5));
interact(plot_segment_IK4,
     x5  = FloatSlider(value=x_5, min=-0.1, max=1, step=0.01, description='x5:'),
     y5  = FloatSlider(value=y_5, min=-0.1, max=1, step=0.01, description='y5:'),
     theta4g=IntSlider(value=theta4_g, min=-180, max=180, step=1, description='θ4:'),
     alphag =IntSlider(value=alpha_g,  min=-180, max=180, step=1, description='α:'))


interactive(children=(FloatSlider(value=0.8, description='x5:', max=1.0, min=-0.1, step=0.01), FloatSlider(val…

<function __main__.plot_segment_IK4(x5, y5, theta4g, alphag)>